# 🚀 TASK 6.3-R: ANCHORED RECOVERY TRAINING TRÊN KAGGLE GPU
### CHIẾN DỊCH HUẤN LUYỆN NEO PHỤC HỒI HIỆU NĂNG ĐỈNH CAO — PHASE 6-R (100% OFFLINE KAGGLE)

> **Mã nhiệm vụ**: `Task 6.3-R` *(Milestone 6-R: Anchored Recovery)*  
> **Đơn vị chủ trì**: **Thành viên C (Training & Evaluation Lead)**  
> **Đơn vị phối hợp**: **Thành viên B (Architecture Lead)**, **Thành viên A (Data & Infra Lead)**  
> **Môi trường thực thi**: Kaggle Server GPU (NVIDIA T4 x2 / P100 / RTX PRO 6000 Blackwell)  
> **Nguyên lý khôi phục**:  
> 1. **Khởi tạo mỏ neo Student từ Checkpoint gốc CVPR 2024**: Khởi tạo từ `fatformer_4class_ckpt.pth` (1.116 keys đạt 96.45% Clean ACC, **tuyệt đối không dùng WiSE làm init**).  
> 2. **Bảo tồn Step 0 Zero-Init Identity**: Tầng chiếu SRM = 0, Cổng tần số = 1.0 (Sai số logit Step 0 = 0.0000000000e+00 bit-perfect).  
> 3. **Tốc độ học phân tầng (Differential Learning Rate)**: Module mới (SRM, Gating) học $lr_2 = 1.0 \times 10^{-4}$; các adapter tiền huấn luyện (FAA, LGA, Text ctx) giảm $10\times$ còn $lr_1 = 1.0 \times 10^{-5}$ để bảo toàn biên độ phân loại sạch; đóng băng 94.1% CLIP ViT-L/14 backbone.  
> 4. **Hàm mất mát neo chặt Teacher**: $\mathcal{L} = 0.3\mathcal{L}_{\text{AugCE}} + 0.7\mathcal{L}_{\text{KD}} + 0.2\mathcal{L}_{\text{cons}}$.  
> 5. **Luồng dữ liệu Clean-First**: Tỷ lệ 70% ảnh sạch / 30% suy biến vật lý nhẹ có kiểm soát.

---
### 📌 HƯỚNG DẪN CẤU HÌNH KAGGLE NOTEBOOK:
1. **Accelerator**: Chọn **GPU T4 x2** hoặc **GPU P100** trong panel bên phải (Settings -> Accelerator).
2. **Input Datasets** (Nhấn 'Add Input' ở góc trên bên phải):
   - `fatformer-code`: Gói mã nguồn dự án (chứa `fatformer_code.zip` vừa đóng gói).
   - `fatformer-assets-pack`: Chứa `ViT-L-14.pt`, `fatformer_4class_ckpt.pth`.
   - `diffusion-staging` (hoặc tập dữ liệu huấn luyện đa miền): Chứa `diffusion_staging.tar`.
3. **Internet**: Có thể BẬT hoặc TẮT (Notebook được thiết kế tương thích 100% Offline).

## 🛠️ BƯỚC 1: THIẾT LẬP MÃ NGUỒN VÀ TỪ VỰNG BPE TOKENIZER TRÊN KAGGLE
> Tự động giải nén mã nguồn từ `fatformer_code.zip` sang `/kaggle/working/fatformer-xla` và chuẩn bị từ vựng BPE offline cho CLIP.

In [ ]:
# ==============================================================================
# BƯỚC 1: THIẾT LẬP MÃ NGUỒN VÀ MÔI TRƯỜNG KAGGLE WORKING
# ==============================================================================
import os, sys, glob, shutil, zipfile, subprocess

WORK_DIR = "/kaggle/working/fatformer-xla"
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

print("=" * 80)
print("[*] BƯỚC 1: NẠP MÃ NGUỒN FATFORMER-XLA VÀO KAGGLE WORKING")
print("=" * 80)

# 1. Dò tìm gói mã nguồn zip trong /kaggle/input/
zip_cands = glob.glob("/kaggle/input/**/fatformer_code.zip", recursive=True) + \
            glob.glob("/kaggle/input/**/fatformer-code*.zip", recursive=True)
train_py_files = glob.glob("/kaggle/input/**/train_crossdomain.py", recursive=True)

if zip_cands:
    src_zip = zip_cands[0]
    print(f"[*] Giải nén mã nguồn từ: {src_zip}")
    with zipfile.ZipFile(src_zip, 'r') as zf:
        zf.extractall(WORK_DIR)
elif train_py_files:
    root_cand = os.path.abspath(os.path.join(os.path.dirname(train_py_files[0]), "..", ".."))
    print(f"[*] Sao chép mã nguồn từ thư mục: {root_cand}")
    shutil.copytree(root_cand, WORK_DIR, dirs_exist_ok=True)
else:
    print("[*] Không thấy gói zip trong Input, đang clone trực tiếp từ GitHub...")
    subprocess.run(f"git clone https://github.com/LeeVHoangtk3/fatformer-xla.git {WORK_DIR}", shell=True, check=True)

# 2. Xử lý từ vựng BPE Tokenizer offline cho CLIP
clip_dir = os.path.join(WORK_DIR, "src", "models", "clip")
os.makedirs(clip_dir, exist_ok=True)
target_bpe_gz = os.path.join(clip_dir, "bpe_simple_vocab_16e6.txt.gz")

bpe_candidates_gz = glob.glob("/kaggle/**/bpe_simple_vocab_16e6.txt.gz", recursive=True) + \
                    glob.glob(f"{WORK_DIR}/**/bpe_simple_vocab_16e6.txt.gz", recursive=True)
bpe_candidates_txt = glob.glob("/kaggle/**/bpe_simple_vocab_16e6.txt", recursive=True)

if not os.path.exists(target_bpe_gz):
    if bpe_candidates_gz:
        shutil.copy2(bpe_candidates_gz[0], target_bpe_gz)
        print(f"[✓] Đã nạp BPE vocab .gz từ: {bpe_candidates_gz[0]}")
    elif bpe_candidates_txt:
        subprocess.run(f"gzip -c '{bpe_candidates_txt[0]}' > '{target_bpe_gz}'", shell=True, check=True)
        print(f"[✓] Đã nén và nạp BPE vocab từ: {bpe_candidates_txt[0]}")

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

try:
    import ftfy
except ImportError:
    subprocess.run('pip install -q ftfy regex tqdm tabulate pyyaml matplotlib --timeout 5', shell=True)
print(f"[✓] Thư mục làm việc hiện tại: {os.getcwd()}")
print("[✓] Hoàn tất Bước 1: Môi trường sẵn sàng 100%!")


## 🔍 BƯỚC 2: ĐỊNH VỊ VÀ LIÊN KẾT TRỌNG SỐ MỎ NEO & CLIP BACKBONE
> Tìm kiếm các trọng số từ `/kaggle/input/` và liên kết vào đúng vị trí chuẩn mực:
> - **CLIP Backbone**: `ViT-L-14.pt` $\rightarrow$ `DATASET/pretrained/ViT-L-14.pt`
> - **Teacher Checkpoint**: `fatformer_4class_ckpt.pth` $\rightarrow$ `DATASET/pretrained/fatformer_4class_ckpt.pth`
> - **Student Init**: `fatformer_4class_ckpt.pth` (Khởi tạo mỏ neo từ bài báo gốc CVPR 2024, **không dùng WiSE**)

In [ ]:
# ==============================================================================
# BƯỚC 2: LIÊN KẾT CHECKPOINT GỐC CVPR 2024 VÀ CLIP BACKBONE
# ==============================================================================
import os, glob, shutil

os.makedirs(os.path.join(WORK_DIR, "DATASET", "pretrained"), exist_ok=True)
os.makedirs(os.path.join(WORK_DIR, "DATASET", "checkpoints"), exist_ok=True)

def find_file(patterns):
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return os.path.abspath(matches[0])
    return None

print("=" * 80)
print("[*] BƯỚC 2: ĐỊNH VỊ VÀ LIÊN KẾT TÀI NGUYÊN CHECKPOINT")
print("=" * 80)

# 1. Dò tìm CLIP ViT-L-14.pt
clip_src = find_file([
    "/kaggle/input/**/ViT-L-14.pt",
    f"{WORK_DIR}/DATASET/pretrained/ViT-L-14.pt"
])
clip_dst = os.path.join(WORK_DIR, "DATASET", "pretrained", "ViT-L-14.pt")
if clip_src and not os.path.exists(clip_dst):
    try: os.symlink(clip_src, clip_dst)
    except Exception: shutil.copy2(clip_src, clip_dst)
print(f"[1] CLIP Backbone : {clip_src} -> {clip_dst}")

# 2. Dò tìm Checkpoint gốc fatformer_4class_ckpt.pth (Teacher & Student Init)
ckpt_4class_src = find_file([
    "/kaggle/input/**/fatformer_4class_ckpt.pth",
    f"{WORK_DIR}/DATASET/pretrained/fatformer_4class_ckpt.pth"
])
ckpt_4class_dst = os.path.join(WORK_DIR, "DATASET", "pretrained", "fatformer_4class_ckpt.pth")
if ckpt_4class_src and not os.path.exists(ckpt_4class_dst):
    try: os.symlink(ckpt_4class_src, ckpt_4class_dst)
    except Exception: shutil.copy2(ckpt_4class_src, ckpt_4class_dst)

print(f"[2] Teacher Anchor: {ckpt_4class_src} -> {ckpt_4class_dst}")
print(f"[3] Student Init  : {ckpt_4class_dst} (Khoi tao truc tiep tu Checkpoint goc CVPR 2024)")

assert os.path.exists(clip_dst), "LỖI: Chưa tìm thấy ViT-L-14.pt trong /kaggle/input!"
assert os.path.exists(ckpt_4class_dst), "LỖI: Chưa tìm thấy fatformer_4class_ckpt.pth trong /kaggle/input!"
print("[✓] Hoàn tất Bước 2: Trọng số gốc và Backbone đã liên kết chuẩn xác!")


## 📦 BƯỚC 3: GIẢI NÉN DỮ LIỆU HUẤN LUYỆN & QUÉT TẠO MANIFEST CHUẨN
> Tự động dò tìm `diffusion_staging.tar` trong `/kaggle/input/`, giải nén sang `/kaggle/working/dataset_train`, quét toàn bộ ảnh để tạo `DATASET/manifests/data_manifest.json` (chia 85% train / 15% val).

In [ ]:
# ==============================================================================
# BƯỚC 3: TRÍCH XUẤT DỮ LIỆU VÀ TẠO MANIFEST CHUẨN 3 TẬP CÂN BẰNG
# ==============================================================================
import tarfile, json as py_json, random

DATA_DIR = "/kaggle/working/dataset_train"
os.makedirs(DATA_DIR, exist_ok=True)

print("=" * 80)
print("[*] BƯỚC 3: GIẢI NÉN DỮ LIỆU HUẤN LUYỆN VÀ SINH MANIFEST")
print("=" * 80)

tar_staging = find_file([
    "/kaggle/input/**/diffusion_staging.tar",
    "/kaggle/input/**/diffusion-staging.tar"
])
staging_dirs = [d for d in glob.glob("/kaggle/input/**/diffusion_staging", recursive=True) if os.path.isdir(d)]

if tar_staging and len(os.listdir(DATA_DIR)) == 0:
    print(f"[*] Đang giải nén dữ liệu từ: {tar_staging} sang {DATA_DIR}...")
    with tarfile.open(tar_staging, 'r') as tf:
        tf.extractall(DATA_DIR)
    print(f"[✓] Đã giải nén hoàn tất sang: {DATA_DIR}")
elif staging_dirs:
    DATA_DIR = staging_dirs[0]
    print(f"[✓] Nhận diện thư mục dữ liệu có sẵn: {DATA_DIR}")

# Quét ảnh thực tế với cơ chế nhận diện nhãn chính xác tuyệt đối
all_records = []
for root, _, files in os.walk(DATA_DIR):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
            fp = os.path.join(root, f)
            norm = fp.replace('\\', '/').lower()
            parts = norm.split('/')
            # Chỉ kiểm tra thư mục con trực tiếp hoặc tên file, TUYỆT ĐỐI không kiểm tra trên cả chuỗi
            is_fake = any(p in ('1_fake', 'fake', 'ai') for p in parts[-3:])
            is_real = any(p in ('0_real', 'real', 'nature', 'clean') for p in parts[-3:])
            if is_real and is_fake:
                for p in reversed(parts):
                    if p in ('0_real', 'real', 'nature', 'clean'):
                        is_fake = False
                        break
                    elif p in ('1_fake', 'fake', 'ai'):
                        is_real = False
                        break
            if not (is_real or is_fake):
                is_fake = ('1_fake' in norm or '/fake/' in norm)

            all_records.append({
                'file_path': fp,
                'label': 1 if is_fake else 0,
                'generator': 'synthetic' if is_fake else 'real'
            })

reals_found = [r for r in all_records if r['label'] == 0]
fakes_found = [r for r in all_records if r['label'] == 1]
print(f"[✓] Tổng số ảnh quét được trong kho dữ liệu: {len(all_records)}")
print(f"    • Số ảnh Real (label=0): {len(reals_found)}")
print(f"    • Số ảnh Fake (label=1): {len(fakes_found)}")

if len(all_records) > 0:
    assert len(reals_found) > 0 and len(fakes_found) > 0, 'LỖI GÁN NHÃN: Không tìm thấy đủ cả ảnh Real và ảnh Fake!'

manifest_path = os.path.join(WORK_DIR, "DATASET", "manifests", "data_manifest.json")
os.makedirs(os.path.dirname(manifest_path), exist_ok=True)

if len(all_records) > 0:
    rng = random.Random(42)
    rng.shuffle(reals_found)
    rng.shuffle(fakes_found)

    # Chia tỷ lệ phân tầng cân bằng 70% train / 15% val_calibration / 15% val_selection
    n_train_r = int(len(reals_found) * 0.70)
    n_calib_r = int(len(reals_found) * 0.15)
    train_r = reals_found[:n_train_r]
    calib_r = reals_found[n_train_r:n_train_r + n_calib_r]
    select_r = reals_found[n_train_r + n_calib_r:]

    n_train_f = int(len(fakes_found) * 0.70)
    n_calib_f = int(len(fakes_found) * 0.15)
    train_f = fakes_found[:n_train_f]
    calib_f = fakes_found[n_train_f:n_train_f + n_calib_f]
    select_f = fakes_found[n_train_f + n_calib_f:]

    train_list = train_r + train_f
    calib_list = calib_r + calib_f
    select_list = select_r + select_f

    rng.shuffle(train_list)
    rng.shuffle(calib_list)
    rng.shuffle(select_list)

    final_records = []
    for item in train_list:
        entry = dict(item)
        entry['split'] = 'train'
        final_records.append(entry)
    for item in calib_list:
        entry = dict(item)
        entry['split'] = 'val_calibration'
        entry['split_compat'] = 'val'
        final_records.append(entry)
    for item in select_list:
        entry = dict(item)
        entry['split'] = 'val_selection'
        entry['split_compat'] = 'val'
        final_records.append(entry)

    with open(manifest_path, "w", encoding="utf-8") as f_out:
        py_json.dump({
            'metadata': {
                'task': 'Task 6.3-R Staging Manifest Balanced',
                'counts': {
                    'total': len(final_records),
                    'train': len(train_list),
                    'val_calibration': len(calib_list),
                    'val_selection': len(select_list)
                }
            },
            'records': final_records
        }, f_out, indent=2)
    print(f"[✓] Đã tạo manifest chuẩn 3 tập: {manifest_path}")
    print(f"    • Train          : {len(train_list)} ảnh (Real: {len(train_r)}, Fake: {len(train_f)})")
    print(f"    • Val Calibration: {len(calib_list)} ảnh (Real: {len(calib_r)}, Fake: {len(calib_f)})")
    print(f"    • Val Selection  : {len(select_list)} ảnh (Real: {len(select_r)}, Fake: {len(select_f)})")
else:
    print("[!] Cảnh báo: Chưa tìm thấy ảnh thực tế, pipeline sẽ sử dụng chế độ dry-run mô phỏng.")


## ⚖️ BƯỚC 4: KIỂM TOÁN GPU & XÁC THỰC TOÁN HỌC ZERO-INIT IDENTITY TẠI STEP 0
> Kiểm tra thiết bị GPU, chạy unit-test xác nhận sai số logit Step 0 đạt mức $0.0000000000\text{e}+00$ bit-perfect và chạy dry-run 1 step kiểm thử autograd.

In [ ]:
# ==============================================================================
# BƯỚC 4: KIỂM TRA GPU VÀ XÁC THỰC ZERO-INIT IDENTITY
# ==============================================================================
import torch, subprocess

print("=" * 80)
print("[*] BƯỚC 4: KIỂM TOÁN PHẦN CỨNG GPU VÀ UNIT-TEST ZERO-INIT")
print("=" * 80)

assert torch.cuda.is_available(), 'LỖI: Chưa bật GPU! Hãy vào Settings -> Accelerator -> Chọn GPU T4 x2 hoặc P100.'
gpu_name = torch.cuda.get_device_name(0)
gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"[✓] GPU khả dụng: {gpu_name} ({gpu_vram:.2f} GB VRAM)")

# 1. Chạy xác thực Zero-Init Step 0 Identity
zi_cmd = f"{sys.executable} tools/test_zero_init_identity.py --device cuda"
print(f"[*] Kiểm tra toán học Step-0 Identity: {zi_cmd}")
res_zi = subprocess.run(zi_cmd, shell=True)
assert res_zi.returncode == 0, "LỖI: Zero-Init test thất bại!"
print("[✓] TOÁN HỌC ZERO-INIT PASS 100%! BẢO TOÀN BIT-PERFECT BIỂU DIỄN SẠCH GỐC.")

# 2. Chạy dry-run autograd
dry_cmd = f"{sys.executable} src/training/train_crossdomain.py --dry_run --device cuda"
res_dry = subprocess.run(dry_cmd, shell=True)
assert res_dry.returncode == 0, "LỖI: Autograd dry-run thất bại!"
print("[✓] SMOKE TEST AUTOGRAD PASS! HỆ THỐNG SẴN SÀNG CHO VÒNG HUẤN LUYỆN CHÍNH THỨC.")


## 🏃 BƯỚC 5: THỰC THI CHIẾN DỊCH HUẤN LUYỆN ANCHORED RECOVERY M6-R (3 EPOCHS)
> Chạy huấn luyện chính thức với đầy đủ các tham số kỹ thuật bất biến theo chỉ đạo của bạn:
> - `--branch M4`: Mô hình đầy đủ SRM + Dynamic Gating + Distillation
> - `--epochs 3`: 3 epochs huấn luyện
> - `--checkpoint_init DATASET/pretrained/fatformer_4class_ckpt.pth`: **Khởi động từ checkpoint gốc (không dùng WiSE)**
> - `--teacher_checkpoint DATASET/pretrained/fatformer_4class_ckpt.pth`: Mỏ neo cố định từ CVPR 2024
> - `--clip_path DATASET/pretrained/ViT-L-14.pt`: Backbone CLIP TorchScript
> - `--weight_ce 0.3 --alpha_kd 0.7 --beta_cons 0.2`: Trọng số hàm mất mát neo chặt Teacher
> - `--learning_rate 1e-4`: Tốc độ học module mới ($10^{-4}$), pretrained adapters tự giảm $10\times$ còn $10^{-5}$
> - `--save_name fatformer_anchored_recovery_best.pth`: Tên checkpoint chuẩn mực

In [ ]:
# ==============================================================================
# BƯỚC 5: CHẠY LỆNH HUẤN LUYỆN CHÍNH THỨC TASK 6.3-R TRÊN KAGGLE GPU
# ==============================================================================
import time, subprocess

CKPT_OUT_DIR = os.path.join(WORK_DIR, "DATASET", "checkpoints")
os.makedirs(CKPT_OUT_DIR, exist_ok=True)

cmd_train = [
    sys.executable, "-u", "src/training/train_crossdomain.py",
    "--branch", "M4",
    "--device", "cuda",
    "--epochs", "3",
    "--batch_size", "16",
    "--checkpoint_init", "DATASET/pretrained/fatformer_4class_ckpt.pth",
    "--teacher_checkpoint", "DATASET/pretrained/fatformer_4class_ckpt.pth",
    "--clip_path", "DATASET/pretrained/ViT-L-14.pt",
    "--save_dir", CKPT_OUT_DIR,
    "--save_name", "fatformer_anchored_recovery_best.pth",
    "--weight_ce", "0.3",
    "--alpha_kd", "0.7",
    "--beta_cons", "0.2",
    "--learning_rate", "1e-4"
]

if os.path.exists(manifest_path) and len(all_records) > 0:
    cmd_train.extend(["--train_manifest", manifest_path])
else:
    print("[!] Cảnh báo: Không có manifest ảnh thực tế, chạy chế độ dry-run.")
    cmd_train.append("--dry_run")

print("=" * 80)
print(f"[*] Lệnh huấn luyện thực thi: {' '.join(cmd_train)}")
print("=" * 80)

t0 = time.time()
proc = subprocess.Popen(cmd_train, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
elapsed = time.time() - t0

if proc.returncode != 0:
    raise RuntimeError(f"[X] Huấn luyện thất bại với mã lỗi: {proc.returncode}")
else:
    print(f"\n[✓] HUẤN LUYỆN ANCHORED RECOVERY HOÀN TẤT THÀNH CÔNG TRONG {elapsed:.2f}s ({elapsed/60:.2f} phút)!")


## 🔐 BƯỚC 6: XÁC NHẬN CHECKPOINT, TÍNH MÃ BĂM SHA-256 & XUẤT RA KAGGLE OUTPUT
> Kiểm tra file `DATASET/checkpoints/fatformer_anchored_recovery_best.pth`, tính mã băm SHA-256, sinh metadata JSON và copy trực tiếp ra `/kaggle/working/` để hiển thị trong panel Kaggle Output tải về.

In [ ]:
# ==============================================================================
# BƯỚC 6: XÁC THỰC CHECKPOINT VÀ XUẤT RA KAGGLE OUTPUT
# ==============================================================================
import hashlib, json as py_json

best_ckpt = os.path.join(CKPT_OUT_DIR, "fatformer_anchored_recovery_best.pth")
assert os.path.exists(best_ckpt), f"LỖI: Không tìm thấy checkpoint tại {best_ckpt}!"

size_mb = os.path.getsize(best_ckpt) / (1024**2)
print(f"[*] Đang tính toán mã băm SHA-256 cho: {best_ckpt} ({size_mb:.2f} MB)...")
sha256 = hashlib.sha256()
with open(best_ckpt, 'rb') as f:
    while chunk := f.read(10 * 1024 * 1024):
        sha256.update(chunk)
digest = sha256.hexdigest().upper()
print(f"[✓] SHA-256 DIGEST: {digest}")

meta_dict = {
    "checkpoint_name": "fatformer_anchored_recovery_best.pth",
    "task": "Task 6.3-R Anchored Recovery Training",
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "sha256": digest,
    "size_bytes": os.path.getsize(best_ckpt),
    "hyperparameters": {
        "branch": "M4",
        "epochs": 3,
        "learning_rate": 1e-4,
        "weight_ce": 0.3,
        "alpha_kd": 0.7,
        "beta_cons": 0.2,
        "checkpoint_init": "DATASET/pretrained/fatformer_4class_ckpt.pth",
        "teacher_checkpoint": "DATASET/pretrained/fatformer_4class_ckpt.pth"
    }
}

meta_path = os.path.join(CKPT_OUT_DIR, "fatformer_anchored_recovery_best_metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    py_json.dump(meta_dict, f, indent=2)

# Sao chép ra thư mục gốc /kaggle/working/ để tải về từ panel Output
out_ckpt = "/kaggle/working/fatformer_anchored_recovery_best.pth"
out_meta = "/kaggle/working/fatformer_anchored_recovery_best_metadata.json"
shutil.copy2(best_ckpt, out_ckpt)
shutil.copy2(meta_path, out_meta)

print(f"[✓] ĐÃ XUẤT CHECKPOINT RA KAGGLE OUTPUT: {out_ckpt} ({os.path.getsize(out_ckpt)/(1024**2):.2f} MB)")
print(f"[✓] ĐÃ XUẤT METADATA SHA-256 RA KAGGLE OUTPUT: {out_meta}")
print("\n🎉 HOÀN TẤT TOÀN DIỆN HUẤN LUYỆN TASK 6.3-R TRÊN KAGGLE GPU!")


## 🎯 BƯỚC 7 (TÙY CHỌN): CHẠY LUÔN HIỆU CHUẨN XÁC SUẤT TASK 6.4-R TRÊN KAGGLE
> Khóa ngưỡng chuẩn $T^*=0.50$, đo đạc nhiệt độ tối ưu $T$ và chấm điểm Gate 4 điều kiện trên tập `val_calibration` ngay tại máy chủ Kaggle.

In [ ]:
# ==============================================================================
# BƯỚC 7: HIỆU CHUẨN XÁC SUẤT TASK 6.4-R VỚI CHECKPOINT VỪA TẠO
# ==============================================================================
cmd_calib = [
    sys.executable, "-u", "tools/calibrate_and_select.py",
    "--checkpoint", best_ckpt,
    "--clip_path", "DATASET/pretrained/ViT-L-14.pt",
    "--manifest", manifest_path,
    "--lock_standard_threshold",
    "--output_dir", "/kaggle/working/",
    "--device", "cuda"
]

print("=" * 80)
print(f"[*] Lệnh hiệu chuẩn Task 6.4-R: {' '.join(cmd_calib)}")
print("=" * 80)

res_calib = subprocess.run(' '.join(cmd_calib), shell=True)
if res_calib.returncode == 0:
    print("\n[✓] HOÀN TẤT HIỆU CHUẨN TASK 6.4-R TRÊN KAGGLE!")
else:
    print("[!] Lưu ý: Hiệu chuẩn có cảnh báo, bạn có thể chạy lại ở Bước 5 Colab.")
